# 03 · Preprocessing Pipeline & Baseline

Turns the preprocessing plan (`docs/03_preprocessing_plan.md`) into code and sets the **baseline** every model must beat.

The shared code lives in `src/clean.py` (fixed rules) and `src/prep.py` (split, preprocessors, metrics), so every notebook uses the same split and the same transformers.

In [1]:
import sys, json
sys.path.append('../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from clean import load_attempts
from prep import (split, linear_preprocessor, tree_preprocessor, evaluate, save_result, to_X,
                  RANDOM_STATE, TEST_START, FEATURES, NUMERIC, CATEGORICAL, LINEAR_NUMERIC, LINEAR_CATEGORICAL)

sns.set_theme(style='whitegrid')
attempts = load_attempts()
X_train, X_test, y_train, y_test = split(attempts)
print('train', X_train.shape, f'make rate {y_train.mean():.1%}')
print('test ', X_test.shape, f'make rate {y_test.mean():.1%}')

train (300000, 22) make rate 74.4%
test  (200000, 22) make rate 78.1%


## 1. Fixed-rule steps (safe before the split)

`load_attempts()` cleans the raw file, builds each lifter's history from **earlier meets only**, reshapes to one row per attempt,
caps extreme values at fixed limits and adds bins with **fixed cut points from the EDA**.
None of these learn anything from the data (no means, medians or fitted values), so they cannot leak test information.

In [2]:
attempts[['lift', 'attempt_no', 'weight', 'prev_best', 'pct_prev_best', 'pct_best_bin', 'prev_result', 'jump_pct', 'jump_bin', 'made']].head(8)

,lift,attempt_no,weight,prev_best,pct_prev_best,pct_best_bin,prev_result,jump_pct,jump_bin,made
0,Squat,1,70.0,NaN,NaN,no history,none,NaN,first attempt,1
1,Squat,1,140.0,NaN,NaN,no history,none,NaN,first attempt,1
2,Squat,1,102.5,NaN,NaN,no history,none,NaN,first attempt,1
3,Squat,1,220.0,NaN,NaN,no history,none,NaN,first attempt,1
4,Squat,1,150.0,NaN,NaN,no history,none,NaN,first attempt,1
5,Squat,1,200.0,NaN,NaN,no history,none,NaN,first attempt,0
6,Squat,1,205.0,NaN,NaN,no history,none,NaN,first attempt,0
7,Squat,1,185.0,NaN,NaN,no history,none,NaN,first attempt,1


In [3]:
for col in ['pct_best_bin', 'jump_bin']:
    print(attempts.groupby(col, observed=True)['made'].agg(['count', 'mean']).round(3), '\n')

                count   mean
pct_best_bin                
100-102.5      385513  0.672
102.5-105      445581  0.658
105-110        481751  0.638
110+           467560  0.614
85-90          340200  0.860
90-95          723759  0.859
95-100        1271729  0.780
<85            554470  0.834
no history    1816351  0.758 



                 count   mean
jump_bin                     
0-2.5           338328  0.552
10+             346206  0.734
2.5-5          1388845  0.669
5-7.5          1198402  0.754
7.5-10          556183  0.758
<=0             456013  0.473
first attempt  2198059  0.900
unknown           4878  0.855 



## 2. Train / test split by meet date

**Train:** meets before 2024. **Test:** meets from 2024 onward. Then a random sample of each side (300,000 / 200,000 attempts).

Why not a random `train_test_split`? Two reasons from the EDA:
1. Attempts from the **same lifter at the same meet** are linked: a random split puts their 2nd squat in train and their 3rd squat in test.
2. The make rate **drifts** over time (74.6% before 2024, 78.1% after).

Splitting by date matches real use: learn from past meets, predict future ones.

In [4]:
pool_train = attempts[attempts['Date'] < TEST_START]
pool_test = attempts[attempts['Date'] >= TEST_START]
print(f'train pool: {len(pool_train):,} attempts, {pool_train["Date"].min().date()} to {pool_train["Date"].max().date()}')
print(f'test pool : {len(pool_test):,} attempts, {pool_test["Date"].min().date()} to {pool_test["Date"].max().date()}')

train_meets = set(zip(attempts.loc[X_train.index, 'lifter_id'], attempts.loc[X_train.index, 'Date']))
test_meets = set(zip(attempts.loc[X_test.index, 'lifter_id'], attempts.loc[X_test.index, 'Date']))
print('lifter-meets in both train and test:', len(train_meets & test_meets))

train_lifters = set(attempts.loc[X_train.index, 'lifter_id'])
seen = attempts.loc[X_test.index, 'lifter_id'].isin(train_lifters).mean()
print(f'test attempts by a lifter who also appears in the training sample: {seen:.1%}')

train pool: 4,902,691 attempts, 1973-11-09 to 2023-12-30
test pool : 1,584,223 attempts, 2024-01-04 to 2026-10-13


lifter-meets in both train and test: 0
test attempts by a lifter who also appears in the training sample: 33.8%


No lifter-meet is in both sets. Some lifters appear in both (a lifter who competed in 2022 and in 2025): that is fine and realistic,
because their history features only ever use **earlier** meets, never the test meet itself.

In [5]:
print('Result columns in the features?', [c for c in ['Best3SquatKg', 'TotalKg', 'Place', 'Goodlift'] if c in X_train.columns])
print(len(FEATURES), 'input columns:', FEATURES)

Result columns in the features? []
22 input columns: ['weight', 'weight_to_bw', 'pct_prev_best', 'jump_kg', 'jump_pct', 'Age', 'BodyweightKg', 'n_prior_meets', 'days_since_last', 'prior_make_rate', 'attempt_no', 'misses_this_lift', 'misses_earlier_lifts', 'lift', 'prev_result', 'Sex', 'Equipment', 'Event', 'tested', 'meet_country', 'pct_best_bin', 'jump_bin']


## 3. Two preprocessors

| | Linear preprocessor (Logistic Regression) | Tree preprocessor (Decision Tree, Random Forest, Gradient Boosting) |
| --- | --- | --- |
| weight_to_bw, Age, BodyweightKg, history counts, misses | median + missing flag, `StandardScaler` | median + missing flag |
| pct_prev_best, jump_pct | as fixed bins → one-hot (incl. "no history", "first attempt") | raw + missing flag |
| weight, jump_kg | left out (correlated with weight_to_bw / jump_pct) | raw |
| attempt_no | one-hot | raw (1, 2, 3) |
| lift, prev_result, Sex, Equipment, Event, tested, meet_country | one-hot (`drop='first'`) | one-hot |

Trees split on thresholds, so they need no scaling or bins. Both are `ColumnTransformer`s that go inside a `Pipeline`, so they are fit on training data only.

In [6]:
lin = linear_preprocessor().fit(X_train)
tree = tree_preprocessor().fit(X_train)
Xl = pd.DataFrame(lin.transform(X_train), columns=lin.get_feature_names_out())
Xt = pd.DataFrame(tree.transform(X_train), columns=tree.get_feature_names_out())
print('linear features:', Xl.shape[1], '| tree features:', Xt.shape[1])
print('missing-value flags added:', [c for c in Xt.columns if c.startswith('missingindicator')])
Xl.head()

linear features: 61 | tree features: 59
missing-value flags added: ['missingindicator_weight_to_bw', 'missingindicator_pct_prev_best', 'missingindicator_jump_kg', 'missingindicator_jump_pct', 'missingindicator_Age', 'missingindicator_BodyweightKg', 'missingindicator_days_since_last', 'missingindicator_prior_make_rate']


,weight_to_bw,Age,BodyweightKg,n_prior_meets,days_since_last,prior_make_rate,misses_this_lift,misses_earlier_lifts,missingindicator_weight_to_bw,missingindicator_Age,...,pct_best_bin_no history,jump_bin_10+,jump_bin_2.5-5,jump_bin_5-7.5,jump_bin_7.5-10,jump_bin_<=0,jump_bin_first attempt,jump_bin_unknown,attempt_no_2,attempt_no_3
0,-0.793582,-0.393605,-0.825683,-0.524521,-0.230923,0.098507,-0.360423,-0.651923,-0.03148,-0.437063,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,-0.934154,-1.365745,-1.614752,0.169680,-0.196723,-0.441980,-0.360423,-0.651923,-0.03148,-0.437063,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,1.420950,1.254806,-0.940755,-0.370254,2.733031,1.148098,2.326295,-0.651923,-0.03148,-0.437063,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,0.523860,1.339340,1.301985,0.863880,-0.473169,-0.406167,-0.360423,-0.651923,-0.03148,-0.437063,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,0.010750,-0.562673,-0.304334,-0.524521,-0.230923,0.098507,-0.360423,-0.651923,-0.03148,-0.437063,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


### Did scaling work?

In [7]:
check = pd.DataFrame({'mean_before': X_train[LINEAR_NUMERIC].mean(), 'std_before': X_train[LINEAR_NUMERIC].std(),
                      'mean_after': Xl[LINEAR_NUMERIC].mean(), 'std_after': Xl[LINEAR_NUMERIC].std()})
check.round(2)

,mean_before,std_before,mean_after,std_after
weight_to_bw,1.91,0.70,-0.0,1.0
Age,29.35,12.79,0.0,1.0
BodyweightKg,81.98,21.30,-0.0,1.0
n_prior_meets,6.80,12.96,0.0,1.0
days_since_last,246.11,406.80,-0.0,1.0
prior_make_rate,0.75,0.15,-0.0,1.0
misses_this_lift,0.13,0.37,0.0,1.0
misses_earlier_lifts,0.63,0.96,-0.0,1.0


After `StandardScaler` every numeric input has mean 0 and standard deviation 1, so the Logistic Regression coefficients are comparable.

## 4. Baseline: `DummyClassifier`

In [8]:
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline

baseline = make_pipeline(tree_preprocessor(), DummyClassifier(strategy='most_frequent')).fit(X_train, y_train)
proba = baseline.predict_proba(X_test)[:, 1]
save_result(evaluate('Baseline (always made)', y_test, proba))

,model,threshold,accuracy,balanced_accuracy,precision_made,recall_made,precision_missed,recall_missed,f1_missed,roc_auc
0,Baseline (always made),0.5,0.7807,0.5,0.7807,1.0,0.0,0.0,0.0,0.5


The baseline is **78.1% accurate** on the test meets while catching **zero** misses: balanced accuracy 0.50, recall for missed attempts 0, ROC-AUC 0.50.
Every model from here on is judged against these numbers, mainly on **ROC-AUC** (ranking quality) and **recall / F1 for missed attempts**.

## 5. How much would a random split flatter the score?

A quick check with a simple Logistic Regression (default settings) on the same number of rows: random `train_test_split` vs the date split.

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

pool = attempts[attempts['Date'] < TEST_START].sample(500_000, random_state=RANDOM_STATE)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(to_X(pool), pool['made'], test_size=0.4,
                                                        stratify=pool['made'], random_state=RANDOM_STATE)
lr = make_pipeline(linear_preprocessor(), LogisticRegression(max_iter=2000))
auc_random = roc_auc_score(yr_test, lr.fit(Xr_train, yr_train).predict_proba(Xr_test)[:, 1])
auc_date = roc_auc_score(y_test, lr.fit(X_train, y_train).predict_proba(X_test)[:, 1])
print(f'ROC-AUC with a random split (meets before 2024): {auc_random:.4f}')
print(f'ROC-AUC with the date split (test from 2024)   : {auc_date:.4f}')

ROC-AUC with a random split (meets before 2024): 0.7658
ROC-AUC with the date split (test from 2024)   : 0.7635


The random split scores **0.766** and the date split **0.764**: here the random split flatters the score only a little (0.002),
so the attempts are not as tightly linked as feared. The date split stays, because it is slightly stricter and it is the honest test of how the model would do at **future** meets.